In [4]:
!pip install huggingface_hub safetensors transformers torch python-dotenv -q

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os
load_dotenv("/home/sagemaker-user/.env")

token = os.environ["HF_TOKEN"]
login(token)

Download model if not already downloaded

In [28]:
from huggingface_hub import snapshot_download

model_id = "Qwen/Qwen2.5-7B-Instruct"  # replace with your target model
model_dir = None
if os.path.exists(f"/home/sagemaker-user/{model_id.split('/')[-1]}"):
    model_dir = f"/home/sagemaker-user/{model_id.split('/')[-1]}"
else:
    model_dir = snapshot_download(
      repo_id=model_id,
    
      local_dir=f"/home/sagemaker-user/{model_id.split('/')[-1]}",  # saves to /content/Qwen2.5-7B-Instruct
      ignore_patterns=["*.md", "*.txt"],  # skip readme files to save time
    )
    
    print(f"Model downloaded to: {model_dir}")

In [ ]:
from safetensors import safe_open
import json

shard_files = sorted([f for f in os.listdir(model_dir) if f.endswith('.safetensors')])
layer_tensors = []

for shard in shard_files:
  with safe_open(os.path.join(model_dir, shard), framework='pt', device='cpu') as f:
    layer_tensors.extend(f.keys())

print(layer_tensors)

In [ ]:
tensor_map = {}

for shard in shard_files:
  with safe_open(os.path.join(model_dir, shard), framework='pt', device='cpu') as f:
    for key in f.keys():
      tensor_map[key] = f.get_tensor(key)

print(f"Loaded {len(tensor_map)} tensors total")

In [ ]:
# The layers you want to duplicate (the "thinking" layers)
layers_to_duplicate = list(range(12, 16))  # layers 12-16

# Build the new layer order: original layers, but with the target range inserted again after itself
# e.g., [0..10, 12..16, 12..16, 17..28]
original_order = list(range(num_layers))
insert_after = layers_to_duplicate[-1]  # insert copies right after the originals

new_layer_order = (
    original_order[:insert_after + 1]       # layers 0..26 (includes the originals)
    + layers_to_duplicate                   # duplicate of layers 20..26
    + original_order[insert_after + 1:]     # layers 27..79
)

print(f"Original layer count: {num_layers}")
print(f"New layer count: {len(new_layer_order)}")
print(f"New order (first 30): {new_layer_order[:30]}")

In [ ]:
new_tensor_map = {}
old_idx_set = set()

for new_idx, old_idx in enumerate(new_layer_order):
  for tensor_name in layer_map[old_idx]:
    new_name = re.sub(
        r"model\.layers\.(\d+)\.",
        f"model.layers.{new_idx}.",
        tensor_name
    )
    if old_idx in old_idx_set:
      new_tensor_map[new_name] = tensor_map[tensor_name].clone()
    else:
      new_tensor_map[new_name] = tensor_map[tensor_name]

  old_idx_set.add(old_idx)

for key, tensor in tensor_map.items():
  if not re.match(r"model\.layers\.\d+\.", key):
      print(f"non_layer params: KEY->{key} TENSOR->{tensor}")
      new_tensor_map[key] = tensor

In [ ]:
del tensor_map
import gc; gc.collect()

In [ ]:
config_path = os.path.join(model_dir, "config.json")
with open(config_path) as f:
    config = json.load(f)

new_num_layers = len(new_layer_order)
config["num_hidden_layers"] = new_num_layers

print(f"Updated num_hidden_layers: {config['num_hidden_layers']}")

In [ ]:
from safetensors.torch import save_file
import math

print(model_id)
new_output_dir = f"/home/sagemaker-user/{model_id.split("/")[-1]}-RYS"
os.makedirs(new_output_dir, exist_ok=True)

SHARD_SIZE = 3 * 1024**3
shard_size = 0
current_shard = {}
new_shards = []

for key, tensor in new_tensor_map.items():
    tensor_size = tensor.element_size() * tensor.numel()

    if shard_size + tensor_size > SHARD_SIZE and current_shard:
        new_shards.append(current_shard)
        current_shard = {}
        shard_size = 0

    current_shard[key] = tensor
    shard_size += tensor_size

if current_shard:
    new_shards.append(current_shard)

In [ ]:
weight_map = {}

for shard_idx, shard_dict in enumerate(new_shards):
  shard_filename = f"model-{shard_idx+1:05d}-of-{len(new_shards):05d}.safetensors"
  shard_path = os.path.join(new_output_dir, shard_filename)
  save_file(shard_dict, shard_path)

  for key in shard_dict:
    weight_map[key] = shard_filename

  print(f"Saved shard {shard_filename}")

index = {"metadata": {"total_size": sum(t.numel() * t.element_size() for t in new_tensor_map.values())},
         "weight_map": weight_map}

with open(os.path.join(new_output_dir, "model.safetensors.index.json"), "w") as f:
    json.dump(index, f, indent=2)

print("Saved index file.")

In [ ]:
with open(f"{new_output_dir}/model.safetensors.index.json") as f:
    index = json.load(f)

in_tensor_map = set(new_tensor_map.keys())
in_index = set(index["weight_map"].keys())

print("In new_tensor_map but MISSING from index:")
print(in_tensor_map - in_index)

print("\nIn index but NOT in new_tensor_map:")
print(in_index - in_tensor_map)

In [ ]:
import shutil
# Copy tokenizer and other config files
files_to_copy = [
    "tokenizer.json", "tokenizer_config.json",
    "special_tokens_map.json", "generation_config.json",
    "tokenizer.model",  # only present in some models
]

for fname in files_to_copy:
    src = os.path.join(model_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(new_output_dir, fname))
        print(f"Copied: {fname}")

# Save the updated config
with open(os.path.join(new_output_dir, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"\nDone! New model saved to: {new_output_dir}")
print(f"  Original layers: {num_layers}")
print(f"  Duplicated layers: {layers_to_duplicate}")
print(f"  New total layers: {new_num_layers}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Loading modified model for sanity check...")
tokenizer = AutoTokenizer.from_pretrained(new_output_dir)
model = AutoModelForCausalLM.from_pretrained(
    new_output_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

prompt = "What is the capital of France?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
     